## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

In this sci-fi setting, the capital of The Moon is Lunaris Prime.

- Location: On the sunlit rim of Shackleton Crater in the south polar region, a hub built to harness solar energy and keep people out of the extreme cold of the crater floor.
- Government seat: Houses the Lunar Council and the Assembly of the Regolith, plus the Lunar Archive and the Grand Observatory.
- Landmarks: Crescent Gate crossing the rim, the Halo Dome for diplomacy, and Sunfall Square where daytime solar arrays glow like a field of stars.
- Architecture and vibe: A gleaming mix of basalt spires and transparent domes, microgravity-boosted plazas, and solar ribbons that shimmer across the skyline. The city thrives on water ice mined in nearby craters and refined into life-support systems, with a bustling transit network linking domed districts.
- Culture and economy: Center of lunar governance, science, and trade—especially in water and helium-3, with a strong emphasis on archival science, space law, and interplan

## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x000002BFC94790C0>, 'last_modified': datetime.datetime(2026, 10, 30, 2, 19, 32, 972000, tzinfo=datetime.timezone.utc)},)


In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [7]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Here’s a portrait of the capital you’re looking at, a city carved into a harsh but beautiful frontier world. I’m calling the city Caelum Prime, the seat of the Asterian Union on the desert world of Sirocco. Its silhouette against the jagged rocks and the distant blue planet is a symbol of resilience and aspiration.

Caelum Prime in a nutshell
- Name and role: Caelum Prime, capital of the Asterian Union. It’s the political heart, cultural beacon, and strategic fulcrum of the system.
- Setting: A basin wedged among basalt spires and weathered mesas. The ground glitters with mineral dust, and narrow canyons funnel wind-sculpted ice-blue pools that glow faintly at night. A colossal blue world looms in the sky, a constant reminder of the wider cosmos and the Union’s ambitions beyond their world.
- Climate and mood: Arid days with sharp temperature swings; nights feel electric as energy networks hum through the rock. The city radiates a sense of disciplined optimism—glass and steel catching 

## Audio input

In [10]:
pip install sounddevice scipy tqdm langchain-core

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 8.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

from langchain_core.messages import HumanMessage

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.68it/s]

Done.


In [12]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Please upload or provide the audio file you would like me to analyze, and let me know what you’d like to learn about it. For example, I can help identify background sounds, speech patterns, or general content. However, I won’t be able to identify or confirm who is speaking from the audio. Let me know how you’d like to proceed.
